# Phase 2 — QGFD vs Softmax: 5-Seed Variance Test

**Goal:** Turn the single-seed result (Δ = −0.0008) into a defensible claim with confidence intervals.

**Config:** 5 seeds (42–46) × {baseline, QGFD α=0.10, diffusion_steps=3} × 300 steps

**Budget:** ~2.5 GPU-hr on Kaggle P100 / 2×T4

**Outputs:**
- CSV with full-precision eval_loss per seed/arm
- Paired Δ: mean, std, 95% CI (t-distribution)
- Wilcoxon signed-rank p-value (nonparametric cross-check)
- it/s ratio stability across seed pairs

---

## 0. Install & Setup

In [ ]:
%%capture
!pip install -q git+https://github.com/rajboopathiking/TorchDire.git
!pip install -q peft trl datasets bitsandbytes accelerate scipy

In [ ]:
import gc
import inspect
import os
import time
import json

import numpy as np
import pandas as pd
import torch
from scipy import stats

from datasets import load_dataset
from peft import LoraConfig
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    set_seed,
)
from trl import SFTConfig, SFTTrainer

from torchdire import (
    patch_llama_with_qgfd,
    register_qgfd_step_callback,
    collect_qgfd_kernels,
)

# Pin single GPU to avoid DataParallel wrapping with bitsandbytes
if "LOCAL_RANK" not in os.environ and "RANK" not in os.environ:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 1. Experiment Configuration

In [ ]:
# ═══════════════════════════════════════════════════
# Phase 2 Config — edit this cell for quick changes
# ═══════════════════════════════════════════════════

MODEL_ID       = "42dot/42dot_LLM-SFT-1.3B"
SEEDS          = [42, 43, 44, 45, 46]
STEPS          = 300
TARGET_ALPHA   = 0.10
WARMUP_STEPS   = 30
DIFFUSION_STEPS = 3
BATCH_SIZE     = 1
GRAD_ACCUM     = 4
LR             = 2e-4
MAX_LENGTH     = 512

# Output paths
OUT_ROOT = "/kaggle/working/phase2"
CSV_PATH = "/kaggle/working/phase2_results.csv"
os.makedirs(OUT_ROOT, exist_ok=True)

print(f"Seeds: {SEEDS}")
print(f"Steps: {STEPS}")
print(f"Config: α={TARGET_ALPHA}, warmup={WARMUP_STEPS}, diffusion_steps={DIFFUSION_STEPS}")
print(f"Budget estimate: {len(SEEDS)} seeds × 2 arms × ~0.25 GPU-hr = ~{len(SEEDS) * 0.5:.1f} GPU-hr")

## 2. Helper Functions

In [ ]:
def format_example(example):
    """Format Alpagasus examples into instruction-following text."""
    if example.get("input"):
        prompt = (
            f"### Instruction:\n{example['instruction']}\n\n"
            f"### Input:\n{example['input']}\n\n### Response:\n{example['output']}"
        )
    else:
        prompt = (
            f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}"
        )
    return {"text": prompt}


def make_sft_config(seed, tag):
    """Build SFTConfig with version-safe max_length handling."""
    kwargs = dict(
        output_dir=os.path.join(OUT_ROOT, f"run-{seed}-{tag}"),
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        max_steps=STEPS,
        num_train_epochs=3,
        learning_rate=LR,
        bf16=True,
        logging_steps=10,
        save_strategy="no",
        dataset_text_field="text",
        packing=False,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        seed=seed,
        data_seed=seed,
    )
    # TRL renamed max_seq_length -> max_length in 1.x
    try:
        fields = SFTConfig.model_fields
    except AttributeError:
        fields = set(inspect.signature(SFTConfig).parameters)
    if "max_length" in fields:
        kwargs["max_length"] = MAX_LENGTH
    elif "max_seq_length" in fields:
        kwargs["max_seq_length"] = MAX_LENGTH
    return SFTConfig(**kwargs)

In [ ]:
def run_single_arm(seed, enable_qgfd, tag):
    """
    Train one arm (baseline or QGFD) for a given seed.
    Returns a dict with eval_loss, train_loss, wall_s, it_per_s.
    """
    set_seed(seed)
    print(f"\n{'='*60}")
    print(f"  Seed={seed}  Arm={tag}  enable_qgfd={enable_qgfd}")
    print(f"{'='*60}")

    # --- Load model ---
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        device_map={"":0},
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype="bfloat16",
        ),
        low_cpu_mem_usage=True,
    )
    model.config.use_cache = False

    # --- QGFD patch ---
    alpha = TARGET_ALPHA if enable_qgfd else 0.0
    warmup = WARMUP_STEPS if enable_qgfd else 0
    model = patch_llama_with_qgfd(
        model,
        diffusion_steps=DIFFUSION_STEPS,
        target_alpha=alpha,
        warmup_steps=warmup,
        early_stop_eps=0.0,
        enable_qgfd=enable_qgfd,
        verbose=True,
    )

    # --- Dataset ---
    ds = load_dataset("arbml/alpagasus_cleaned")["train"].map(format_example)
    ds = ds.train_test_split(test_size=0.2, seed=seed)
    train_ds, test_ds = ds["train"], ds["test"]

    # --- LoRA ---
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )

    # --- Trainer ---
    trainer = SFTTrainer(
        model=model,
        train_dataset=train_ds,
        eval_dataset=test_ds,
        args=make_sft_config(seed, tag),
        peft_config=lora_config,
    )
    trainer.model.print_trainable_parameters()
    register_qgfd_step_callback(trainer, model)

    # --- Verify QGFD kernels ---
    kernels = collect_qgfd_kernels(model)
    print(f"  QGFD kernels found: {len(kernels)}")
    if enable_qgfd and len(kernels) > 0:
        print(f"  enable_qgfd={kernels[0].enable_qgfd}, "
              f"target_alpha={kernels[0].target_alpha}, "
              f"diffusion_steps={kernels[0].diffusion_steps}")

    # --- Train ---
    t0 = time.time()
    trainer.train()
    wall_time = time.time() - t0

    # --- Evaluate ---
    eval_metrics = trainer.evaluate()
    eval_loss = eval_metrics.get("eval_loss", float("nan"))
    train_losses = [l["loss"] for l in trainer.state.log_history if "loss" in l]
    train_loss = train_losses[-1] if train_losses else float("nan")

    # --- Verify alpha after training ---
    if enable_qgfd and len(kernels) > 0:
        final_step = kernels[0].step_count.item()
        final_alpha = kernels[0].get_alpha()
        print(f"  Final step_count={final_step}, alpha_eff={final_alpha:.6f}")

    result = {
        "seed": seed,
        "arm": tag,
        "enable_qgfd": enable_qgfd,
        "alpha": alpha,
        "train_loss": round(train_loss, 6),
        "eval_loss": round(eval_loss, 6),
        "eval_ppl": round(float(torch.exp(torch.tensor(eval_loss))), 4),
        "wall_s": round(wall_time, 1),
        "it_per_s": round(STEPS / wall_time, 3),
    }
    print(f"  Result: eval_loss={result['eval_loss']:.6f}, "
          f"ppl={result['eval_ppl']:.4f}, "
          f"wall={result['wall_s']}s, "
          f"it/s={result['it_per_s']}")

    # --- Cleanup ---
    del model, trainer, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

    return result

## 3. Pre-flight Sanity Check (1 seed, 20 steps)

Quick smoke test before committing GPU-hours. Verifies:
- LoRA adapts `o_proj` (not `out_proj`)
- QGFD diffusion activates (non-zero alpha)
- Memory cleanup works between arms

In [ ]:
# Temporarily override for smoke test
_STEPS_BAK = STEPS
STEPS = 20

smoke_base = run_single_arm(seed=42, enable_qgfd=False, tag="smoke-baseline")
smoke_qgfd = run_single_arm(seed=42, enable_qgfd=True,  tag="smoke-qgfd")

STEPS = _STEPS_BAK  # restore

print(f"\n✅ Smoke test passed.")
print(f"   Baseline: eval_loss={smoke_base['eval_loss']:.6f}")
print(f"   QGFD:     eval_loss={smoke_qgfd['eval_loss']:.6f}")
print(f"   Cost ratio: {smoke_qgfd['wall_s'] / max(smoke_base['wall_s'], 0.1):.2f}x")

## 4. Phase 2 — Full 5-Seed Variance Run

For each seed, we run **baseline first, then QGFD**, with full GPU memory cleanup between each. Results are saved incrementally to CSV so a Kaggle session timeout doesn't lose completed runs.

In [ ]:
all_results = []

# Resume support: if CSV already exists from a previous partial run, load it
completed_keys = set()
if os.path.exists(CSV_PATH):
    prev_df = pd.read_csv(CSV_PATH)
    all_results = prev_df.to_dict("records")
    completed_keys = {(r["seed"], r["arm"]) for r in all_results}
    print(f"Resuming: {len(all_results)} runs already completed.")

total_runs = len(SEEDS) * 2
run_idx = len(all_results)

for seed in SEEDS:
    for enable_qgfd, tag in [(False, "baseline"), (True, "qgfd")]:
        if (seed, tag) in completed_keys:
            print(f"⏭ Skipping seed={seed}, arm={tag} (already completed)")
            continue

        run_idx += 1
        print(f"\n[Run {run_idx}/{total_runs}]")

        result = run_single_arm(seed=seed, enable_qgfd=enable_qgfd, tag=tag)
        all_results.append(result)

        # Incremental save after every run
        pd.DataFrame(all_results).to_csv(CSV_PATH, index=False)
        print(f"  💾 Saved to {CSV_PATH} ({len(all_results)} rows)")

print(f"\n{'='*60}")
print(f"  Phase 2 complete: {len(all_results)} runs saved to {CSV_PATH}")
print(f"{'='*60}")

## 5. Results Table

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} rows from {CSV_PATH}\n")
display(df)

## 6. Statistical Analysis — Paired Difference Test

In [ ]:
df = pd.read_csv(CSV_PATH)

# Pivot: one row per seed, columns for baseline and qgfd eval_loss
base = df[df["arm"] == "baseline"].set_index("seed")["eval_loss"]
qgfd = df[df["arm"] == "qgfd"].set_index("seed")["eval_loss"]

# Paired difference: positive Δ means baseline is worse (QGFD wins)
deltas = base - qgfd
n = len(deltas)

print(f"Seeds: {list(deltas.index)}")
print(f"n = {n}\n")
print("Per-seed paired differences (Δ = baseline − QGFD):")
for seed, d in deltas.items():
    print(f"  Seed {seed}: Δ = {d:+.6f}")

mean_delta = deltas.mean()
std_delta = deltas.std(ddof=1)
se_delta = std_delta / np.sqrt(n)

# 95% CI using t-distribution (appropriate for n=5)
t_crit = stats.t.ppf(0.975, df=n-1)
ci_low = mean_delta - t_crit * se_delta
ci_high = mean_delta + t_crit * se_delta

# Paired t-test
t_stat, p_value_t = stats.ttest_1samp(deltas, 0)

# Wilcoxon signed-rank (nonparametric cross-check)
try:
    w_stat, p_value_w = stats.wilcoxon(deltas, alternative='two-sided')
except ValueError:
    # All differences are zero or n too small
    w_stat, p_value_w = float('nan'), float('nan')

print(f"\n{'='*55}")
print(f"  PAIRED DIFFERENCE ANALYSIS (Δ = baseline − QGFD)")
print(f"{'='*55}")
print(f"  mean(Δ)  = {mean_delta:+.6f}")
print(f"  std(Δ)   = {std_delta:.6f}")
print(f"  SE(Δ)    = {se_delta:.6f}")
print(f"  95% CI   = [{ci_low:+.6f}, {ci_high:+.6f}]")
print(f"  t-stat   = {t_stat:.4f}  (p = {p_value_t:.4f})")
print(f"  Wilcoxon = {w_stat}  (p = {p_value_w:.4f})")
print()

if ci_low > 0:
    print("  ✅ CI excludes 0: QGFD significantly BETTER (lower eval_loss).")
elif ci_high < 0:
    print("  ⚠️  CI excludes 0: QGFD significantly WORSE (higher eval_loss).")
else:
    print("  ➖ CI straddles 0: Δ not distinguishable from noise at 95% level.")
    print("     This is a legitimate, publishable finding — not a failure.")

## 7. Compute Cost Ratio Analysis

In [ ]:
df = pd.read_csv(CSV_PATH)

base_its = df[df["arm"] == "baseline"].set_index("seed")["it_per_s"]
qgfd_its = df[df["arm"] == "qgfd"].set_index("seed")["it_per_s"]

base_wall = df[df["arm"] == "baseline"].set_index("seed")["wall_s"]
qgfd_wall = df[df["arm"] == "qgfd"].set_index("seed")["wall_s"]

cost_ratio = qgfd_wall / base_wall
throughput_ratio = qgfd_its / base_its

print("Per-seed compute cost ratio (QGFD wall_s / baseline wall_s):")
for seed in cost_ratio.index:
    print(f"  Seed {seed}: {cost_ratio[seed]:.3f}x  "
          f"(baseline={base_wall[seed]:.0f}s, QGFD={qgfd_wall[seed]:.0f}s)")

print(f"\nmean cost ratio:       {cost_ratio.mean():.3f}x ± {cost_ratio.std():.3f}")
print(f"mean throughput ratio: {throughput_ratio.mean():.3f}x ± {throughput_ratio.std():.3f}")
print(f"\nBaseline mean it/s: {base_its.mean():.3f} ± {base_its.std():.3f}")
print(f"QGFD     mean it/s: {qgfd_its.mean():.3f} ± {qgfd_its.std():.3f}")

## 8. Summary Plots

In [ ]:
import matplotlib.pyplot as plt

df = pd.read_csv(CSV_PATH)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Plot 1: Eval loss by seed and arm ---
ax = axes[0]
base_vals = df[df["arm"] == "baseline"].sort_values("seed")
qgfd_vals = df[df["arm"] == "qgfd"].sort_values("seed")
x = np.arange(len(SEEDS))
w = 0.35
ax.bar(x - w/2, base_vals["eval_loss"].values, w, label="Baseline", color="#2196F3", alpha=0.85)
ax.bar(x + w/2, qgfd_vals["eval_loss"].values, w, label="QGFD", color="#FF5722", alpha=0.85)
ax.set_xlabel("Seed")
ax.set_ylabel("Eval Loss")
ax.set_title("Eval Loss by Seed")
ax.set_xticks(x)
ax.set_xticklabels(SEEDS)
ax.legend()
ax.grid(axis="y", alpha=0.3)

# --- Plot 2: Paired differences ---
ax = axes[1]
base_loss = df[df["arm"] == "baseline"].set_index("seed")["eval_loss"]
qgfd_loss = df[df["arm"] == "qgfd"].set_index("seed")["eval_loss"]
deltas = base_loss - qgfd_loss
colors = ["#4CAF50" if d > 0 else "#F44336" for d in deltas.values]
ax.bar(range(len(deltas)), deltas.values, color=colors, alpha=0.85)
ax.axhline(y=0, color="black", linewidth=0.8)
ax.axhline(y=deltas.mean(), color="#9C27B0", linewidth=2, linestyle="--",
           label=f"mean Δ = {deltas.mean():+.4f}")
ax.set_xlabel("Seed")
ax.set_ylabel("Δ (baseline − QGFD)")
ax.set_title("Paired Differences")
ax.set_xticks(range(len(deltas)))
ax.set_xticklabels(deltas.index)
ax.legend()
ax.grid(axis="y", alpha=0.3)

# --- Plot 3: Wall time comparison ---
ax = axes[2]
ax.bar(x - w/2, base_vals["wall_s"].values, w, label="Baseline", color="#2196F3", alpha=0.85)
ax.bar(x + w/2, qgfd_vals["wall_s"].values, w, label="QGFD", color="#FF5722", alpha=0.85)
ax.set_xlabel("Seed")
ax.set_ylabel("Wall Time (s)")
ax.set_title("Training Time by Seed")
ax.set_xticks(x)
ax.set_xticklabels(SEEDS)
ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
fig.savefig("/kaggle/working/phase2_plots.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: /kaggle/working/phase2_plots.png")

## 9. Final Summary

Printable summary block for inclusion in reports/papers.

In [ ]:
df = pd.read_csv(CSV_PATH)

base_loss = df[df["arm"] == "baseline"].set_index("seed")["eval_loss"]
qgfd_loss = df[df["arm"] == "qgfd"].set_index("seed")["eval_loss"]
deltas = base_loss - qgfd_loss
n = len(deltas)
mean_d = deltas.mean()
std_d = deltas.std(ddof=1)
se_d = std_d / np.sqrt(n)
t_c = stats.t.ppf(0.975, df=n-1)
_, p_t = stats.ttest_1samp(deltas, 0)

base_wall = df[df["arm"] == "baseline"]["wall_s"]
qgfd_wall = df[df["arm"] == "qgfd"]["wall_s"]
cost = qgfd_wall.values / base_wall.values

print("╔══════════════════════════════════════════════════════════╗")
print("║   PHASE 2 SUMMARY — QGFD vs Softmax, 5-Seed Variance  ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  Model:  {MODEL_ID:<46}  ║")
print(f"║  Config: α={TARGET_ALPHA}, diffusion_steps={DIFFUSION_STEPS}, "
      f"warmup={WARMUP_STEPS}, steps={STEPS}  ║")
print(f"║  Seeds:  {SEEDS}                          ║")
print(f"║                                                        ║")
print(f"║  Baseline eval_loss: {base_loss.mean():.4f} ± {base_loss.std():.4f}          ║")
print(f"║  QGFD    eval_loss: {qgfd_loss.mean():.4f} ± {qgfd_loss.std():.4f}          ║")
print(f"║                                                        ║")
print(f"║  mean(Δ):  {mean_d:+.6f}                              ║")
print(f"║  std(Δ):   {std_d:.6f}                               ║")
print(f"║  95% CI:   [{mean_d - t_c*se_d:+.6f}, {mean_d + t_c*se_d:+.6f}]             ║")
print(f"║  p-value:  {p_t:.4f} (paired t-test)                   ║")
print(f"║                                                        ║")
print(f"║  Cost ratio: {cost.mean():.3f}x ± {cost.std():.3f}x               ║")
print("╚══════════════════════════════════════════════════════════╝")

---

## Decision Rule

- **If the 95% CI excludes zero** → QGFD has a real (if small) quality effect in one direction.
- **If the CI straddles zero** → the result is genuinely indistinguishable from noise — a legitimate, publishable finding, not a failure.

### Next steps based on outcome:

| Outcome | Next Phase |
|---------|------------|
| CI excludes 0 (QGFD wins) | Phase 3 (ablation) to find optimal α/steps |
| CI excludes 0 (QGFD loses) | Phase 3 to check if a different config helps |
| CI straddles 0 | Phase 3 to rule out wrong hyperparams, Phase 4 for longer horizon |